# Phase 0 — P0.1 chạy lại trên ảnh đã xử lý (bản 3, 17/10) · Colab T4

Thứ tự (PG_update mục 2): **xử lý ảnh → QA tự động → QA trực quan (bạn chấm) → trích embedding → 4 chỉ số → báo cáo**.
Hướng dẫn chi tiết từng bước: `phase0/HUONG_DAN_CHAY.md`.

**Trước khi chạy**
- Đã commit + push nhánh `PAG` (notebook clone mã từ GitHub).
- Trên Drive: `MyDrive/p01_images.zip` và `MyDrive/p01_weights/` (tạo bằng `python -m phase0.make_colab_bundle`).
- Runtime → Change runtime type → **T4 GPU**.

**Bảo mật:** tên file ảnh chứa họ tên → không in tên file. Các cell xem ảnh hiển thị phim thật → **trước khi lưu/commit notebook: Edit → Clear all outputs**. Ảnh đã xử lý chỉ để ở Drive riêng tư.

In [ ]:
import torch
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/repo && git clone -q -b PAG https://github.com/Do-Trung-Quan/DO-AN-TOT-NGHIEP-E22-T02.git /content/repo
%cd /content/repo
!pip install -q torchxrayvision opencv-python-headless easyocr hi-ml-multimodal pyarrow
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO GPU — doi runtime sang T4')
!git log -1 --format='%h %s'
!python -c "from phase0.common import load_config; print('config', load_config()['_hash'])"

In [ ]:
# Ảnh gốc + trọng số (không có trong git)
!mkdir -p /content/images
!cp /content/drive/MyDrive/p01_images.zip /content/p01_images.zip
!unzip -q -o /content/p01_images.zip -d /content/images
!echo "so anh (ky vong 1835):" && find /content/images -name '*.jpg' | wc -l
!cp "/content/drive/MyDrive/p01_weights/biovil_t_image_model_proj_size_128.pt" "/content/repo/Pre-train BioViL-T/"
!cp "/content/drive/MyDrive/p01_weights/best_model_ce.pth" "/content/repo/Finetune CrossEntropy/"
!cp /content/drive/MyDrive/p01_weights/phase3_oof_*.npz /content/repo/fusion/output/
PROC = '/content/drive/MyDrive/p01_processed'   # o Drive de chay tiep duoc neu Colab ngat

## Bước 1. Xử lý ảnh — thử 30 ảnh trước
Xem lưới bên dưới: **ô trái** = V1 (ảnh đã cắt + che chữ, viền đỏ là mask phổi), **ô phải** = V2 (chỉ còn phổi).
Kiểm tra: hai phổi nằm trọn trong ảnh (không mất đỉnh phổi/góc sườn hoành), viền đỏ ôm đúng phổi, không còn chữ. Nếu sai nhiều thì dừng lại và báo để sửa script.

In [ ]:
!python -u -m phase0.p01_preprocess --images-root /content/images --out-dir /content/p01_sample --limit 30

In [ ]:
import numpy as np, pandas as pd
from phase0.p01_qa_sample import ve_luoi
z = np.load('/content/p01_sample/p01_processed_lim.npz')
qa = pd.read_csv('/content/p01_sample/p01_mask_qa_lim.csv')
flag_cols = [c for c in qa.columns if c.startswith('co_') and c != 'co_bat_ky']
bang = pd.DataFrame({'stt': np.arange(1, len(qa) + 1), 'img_id': qa.img_id,
                     'co': [';'.join(c[3:] for c in flag_cols if r[c]) for _, r in qa.iterrows()]})
ve_luoi(z, bang, 0, 30)

## Bước 2. Xử lý toàn bộ 1.835 ảnh (~1–1,5 giờ trên T4)
Kết quả ghi theo từng shard 100 ảnh vào Drive; nếu Colab ngắt, chạy lại cell setup + cell này, nó sẽ bỏ qua shard đã xong.
Cuối cell in **bảng QA tự động** (số ảnh bị mỗi loại cờ) — chép bảng này vào báo cáo.

In [ ]:
!python -u -m phase0.p01_preprocess --images-root /content/images --out-dir {PROC}

**Xem ảnh đã xử lý ở đâu:** toàn bộ nằm trong `MyDrive/p01_processed/p01_processed.npz` (mảng số, không mở trực tiếp bằng trình xem ảnh). Dùng cell dưới để xem bất kỳ ảnh nào theo `img_id`: V0 (toàn ảnh đã che chữ) · V1 (cắt + che chữ, viền đỏ = mask phổi) · V2 (chỉ còn phổi).
Mặc định hiện 5 ảnh bị cờ đầu tiên; đổi `XEM` thành danh sách img_id bạn muốn.

In [ ]:
import numpy as np, pandas as pd
from phase0.p01_qa_sample import xem_anh
z = np.load(f'{PROC}/p01_processed.npz')
qa = pd.read_csv(f'{PROC}/p01_mask_qa.csv')
XEM = qa.loc[qa.co_bat_ky == 1, 'img_id'].head(5).tolist()   # vi du: XEM = [38, 281]
xem_anh(z, XEM, qa)

## Bước 3. QA trực quan — bạn chấm
Cell dưới chọn **50 ảnh ngẫu nhiên phân tầng lô × nhãn + mọi ảnh bị cờ**, ghi `p01_qa_cham_tay.csv` vào Drive, rồi vẽ lưới theo trang (mỗi trang 40 ảnh, đánh số `#stt`).

Chấm mỗi ảnh một mức:
- `dat`: hai phổi nằm trọn, mask ôm đúng phổi, không còn chữ;
- `mot_phan`: dùng được nhưng có lỗi nhỏ (cắt hơi rộng/hẹp, mask hụt một góc, còn vệt chữ nhỏ ngoài phổi);
- `hong`: mất một phần phổi, mask sai hẳn, hoặc ảnh không đọc được.

In [ ]:
!python -m phase0.p01_qa_sample --qa {PROC}/p01_mask_qa.csv --out {PROC}/p01_qa_cham_tay.csv
z = np.load(f'{PROC}/p01_processed.npz')
bang = pd.read_csv(f'{PROC}/p01_qa_cham_tay.csv')
print(len(bang), 'anh can cham')

In [ ]:
TRANG = 0          # doi 0, 1, 2, ... de xem trang tiep theo
ve_luoi(z, bang, TRANG * 40, (TRANG + 1) * 40)

Ghi điểm theo **một trong hai cách**:
1. Mở `MyDrive/p01_processed/p01_qa_cham_tay.csv` bằng Google Sheets, điền cột `diem`, tải về dạng CSV và ghi đè đúng tên cũ; hoặc
2. Điền trực tiếp vào cell dưới: chỉ cần ghi các ảnh **không đạt**; ảnh nào không ghi được tính là `dat`.

In [ ]:
KHONG_DAT = {
    # stt: 'mot_phan' hoac 'hong', vi du:
    # 7: 'mot_phan',
    # 23: 'hong',
}
DUNG_CELL_NAY = True    # False neu ban da dien CSV bang Google Sheets (cach 1)
if DUNG_CELL_NAY:
    bang['diem'] = [KHONG_DAT.get(int(s), 'dat') for s in bang.stt]
    bang.to_csv(f'{PROC}/p01_qa_cham_tay.csv', index=False)
print(pd.read_csv(f'{PROC}/p01_qa_cham_tay.csv').diem.value_counts())

In [ ]:
!python -m phase0.p01_qa_summary --qa {PROC}/p01_mask_qa.csv --cham {PROC}/p01_qa_cham_tay.csv --out-dir phase0/output

**Chỉ đi tiếp khi dòng cuối in `CONG QA: DAT`.** Nếu KHÔNG ĐẠT: dừng, ghi lại ảnh hỏng thuộc kiểu gì để sửa `p01_preprocess.py`.

## Bước 4. Trích embedding V0, V1, V2 cho frozen và control (~10 phút)

In [ ]:
for mode in ['frozen', 'control']:
    for v in ['V0', 'V1', 'V2']:
        !python -u -m phase0.p01_extract_embeddings --processed {PROC}/p01_processed.npz --mode {mode} --variant {v} --out-dir {PROC}

## Bước 5. Bốn chỉ số P0.1 (~15–25 phút, CPU)
`A_goc` = đặc trưng cũ, chạy lại để so sánh (phải in `max|diff| = 0.00e+00` và `L1 AUC gop = 0.760` với control).
V1/V2: ảnh chấm `hong` dùng vector V0 và gắn cờ `zone_missing`.

In [ ]:
for s in ['control', 'frozen']:
    !python -u -m phase0.p01_batch_noise --source {s} --tag A_goc
    for v in ['V1', 'V2']:
        !python -u -m phase0.p01_batch_noise --source {s} --tag {v} --features {PROC}/image_features_{s}_{v}.parquet --fallback {PROC}/image_features_{s}_V0.parquet --zone-missing phase0/output/p01_zone_missing.csv

In [ ]:
!python -m phase0.p01_decision

## Bước 6. Lưu kết quả về Drive
Chỉ số liệu (json/md/npz điểm dự đoán, CSV chỉ có img_id) — không có ảnh, không có tên file. Tải thư mục `MyDrive/phase0_results/` về máy, chép vào `phase0/output/` của repo.

In [ ]:
!mkdir -p /content/drive/MyDrive/phase0_results
!cp phase0/output/p01_* /content/drive/MyDrive/phase0_results/
!cp {PROC}/p01_mask_qa.csv {PROC}/p01_qa_cham_tay.csv {PROC}/image_features_*_manifest.json /content/drive/MyDrive/phase0_results/
!ls -la /content/drive/MyDrive/phase0_results/